# dfx4ml quick start — QUANTIZED ResNet-8 / CIFAR-10 (`config_4`: 2 PR regions, 4 RMs)

**Target board: Xilinx KV260** (Zynq UltraScale+, Vivado/Vitis 2023.2, Ubuntu 22.04 + PYNQ).

End-to-end **QKeras → hls4ml → dfx4ml** build for the **4-partition** topology of the
**quantized** ResNet-8 CIFAR-10 model
(`/media/tanawin/tanawin1701e/project7/hls4ml/exp_sz/resnet8_cifar10_q.keras`),
cut into **four** kernels (`part1`..`part4`) across two reconfigurable regions, 2 RMs each.
Conv+BN are fused (`QConv2DBatchnorm`) and weights are uniformly **8-bit** (per-layer
precision auto-derived from the QKeras quantizers).

**Per-layer reuse factors** are rebalanced (vs `exp_sz/hls_cls_cnn_q.py`) so the four kernels
are **as equal as possible in resource usage**. A layer's parallel multipliers ≈
`weights / ReuseFactor` (DSP ≈ 0.64·W/RF, LUT ≈ 31·W/RF), and since weights are 8-bit uniform
the per-partition sum of `W/RF` ("eff") is the area proxy. A region's pblock must fit its
**largest** RM, and phases alternate regions, so region A=max(part1,part3), B=max(part2,part4):

| part | region | layers | eff (Σ W/RF) |
|---|---|---|---|
| part1 | A | conv2d, conv2d_1 | 432 |
| part2 | B | conv2d_2, conv2d_3, conv2d_5 | 416 |
| part3 | A | conv2d_4, conv2d_6 | 448 |
| part4 | B | conv2d_7, conv2d_8, dense | 426 |

→ **region A = 448, region B = 426** (was 576 / 512 with the float RF map).

Run from the **repo root**. The notebook is self-contained: it loads the trained quantized
model, builds the four shared-weight partition sub-models inline, and runs every stage from
the topology `CONFIG` through the hardware + software build, staging the data the board test
notebook (`hls4ml_2_region_4_rm.ipynb`) needs. The final cell tells you how to copy the
packaged project to the KV260.

> Requires **qkeras** in the kernel env (`pip install qkeras`).

## 1. Environment + backend
Add `lib/` and the `hls4ml` submodule to the path, register the dfx4ml backend, and import
the build helpers.

In [1]:
import os, sys, json, shutil
from pathlib import Path

REPO = Path.cwd()                                # run this notebook from the repo root
sys.path.insert(0, str(REPO / 'lib'))            # lib/hls4ml_build + lib/hls4ml_con
sys.path.insert(0, str(REPO / 'hls4ml'))         # hls4ml submodule source
os.environ['HLS4ML_BACKEND_PLUGINS'] = 'hls4ml_con'   # discovered at `import hls4ml`
os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '3')

import numpy as np
import tensorflow as tf
import hls4ml
# QKeras loader for the quantized model (resnet8_cifar10_q.keras). If this kernel's env
# lacks qkeras, install it once:   pip install qkeras
try:
    from qkeras.utils import load_qmodel
except ModuleNotFoundError as e:
    raise SystemExit('qkeras not installed in this env — run `pip install qkeras` '
                     'in the notebook kernel (dfx4ml-hls4ml).') from e
from hls4ml_build import Hls4ml_build, Partition, Stream, DMA   # orchestrator + typed topology
from lib.hw_build import HwBuildHelper
from lib.sw_build import SwBuildHelper

assert 'vitisunifieddfx4ml' in hls4ml.backends.get_available_backends(), \
    'VitisUnifiedDFx4ml backend not registered — check HLS4ML_BACKEND_PLUGINS / sys.path'
print('OK: VitisUnifiedDFx4ml registered')

OK: VitisUnifiedDFx4ml registered


## 2. Load the model + build the 4 partition sub-models
Load the trained **quantized** ResNet-8 (`resnet8_cifar10_q.keras`, via `load_qmodel`) and
build the four partition sub-models **by reusing the full model's layer objects**
(`full.get_layer(...)`) — so every partition shares the trained weights.

In the quantized model conv+BN are **fused** into one `QConv2DBatchnorm` layer (no separate
`batch_normalization_*`), and the ReLUs are `q_activation_*` (`QActivation`). The graph
(stem + 3 residual stages + GAP/Dense), with the cut points marked:

```
in0 ─ conv2d(+BN) ─ relu(=stem) ─ conv2d_1(+BN) ─ relu(=a1) ─┐  part1 → streams: a1, skip0(=stem)
                                                              │
a1 ─ conv2d_2(+BN) ─[+ skip0]─ relu(=s1o) ─┬─ conv2d_3(+BN) ─ relu(=c2) ─┐  part2 → streams: c2, skip2
                                           └─ conv2d_5(+BN) (=skip2) ─────┘
c2 ─ conv2d_4(+BN) ─[+ skip2]─ relu(=s2o=m2) ─ conv2d_6(+BN) ─ relu(=c3) ─┐  part3 → streams: c3, m2(=s2o)
m2 ─ conv2d_8(+BN) (=x8) ───────────────────────────────────────┐        │
c3 ─ conv2d_7(+BN) (=x7) ─[+ x8]─ relu ─ GAP ─ dense ─ out       │ part4 → DMA out
```
(`relu` = `q_activation_*`; `dense` output is logits, pre-softmax.)

In [2]:
# >>> EDIT if your trained model lives elsewhere <<<  (QUANTIZED QKeras model)
MODEL_FILE = Path('/media/tanawin/tanawin1701e/project7/hls4ml/exp_sz/resnet8_cifar10_q.keras')
assert MODEL_FILE.exists(), f'model not found: {MODEL_FILE} (run exp_sz/train_cls_cnn_q.py first)'

full = load_qmodel(str(MODEL_FILE))                 # QKeras loader (registers Q* custom objects)
L    = full.get_layer
inp  = full.input                                   # (32, 32, 3)
print('full model params:', full.count_params())

# NOTE vs the float model: conv+BN are fused into one QConv2DBatchnorm layer (so there are no
# separate `batch_normalization_*` layers), and the ReLUs are `q_activation_*` (QActivation).
# Weights are uniformly 8-bit; per-layer precision is auto-derived from the QKeras quantizers.

# ── part1 ── stem + stage1 conv_a ─────────────  in: DMA → out: a1, skip0(=stem relu)
stem = L('q_activation').output                     # relu(conv2d+BN)    (32,32,16) stage1 skip
a1   = L('q_activation_1').output                   # relu(conv2d_1+BN)  (32,32,16)
part1 = tf.keras.Model(inp, [a1, stem], name='part1')

# ── part2 ── stage1 conv_b + stage2 conv_a + stage2 skip ──  in: a1, skip0 → out: c2, skip2
p2_a1   = tf.keras.Input((32, 32, 16), name='p2_a1')
p2_skip0 = tf.keras.Input((32, 32, 16), name='p2_skip0')
_s1o = L('q_activation_2')(L('add')([L('conv2d_2')(p2_a1), p2_skip0]))   # stage1 out (conv2d_2 = conv+BN)
_c2  = L('q_activation_3')(L('conv2d_3')(_s1o))                          # (16,16,32)
_sk2 = L('conv2d_5')(_s1o)                                               # (16,16,32) skip (conv+BN, no act)
part2 = tf.keras.Model([p2_a1, p2_skip0], [_c2, _sk2], name='part2')

# ── part3 ── stage2 conv_b + stage3 conv_a ──  in: c2, skip2 → out: c3, m2(=stage2 out)
p3_c2  = tf.keras.Input((16, 16, 32), name='p3_c2')
p3_sk2 = tf.keras.Input((16, 16, 32), name='p3_sk2')
_s2o = L('q_activation_4')(L('add_1')([L('conv2d_4')(p3_c2), p3_sk2]))   # stage2 out
_c3  = L('q_activation_5')(L('conv2d_6')(_s2o))                          # (8,8,64)
part3 = tf.keras.Model([p3_c2, p3_sk2], [_c3, _s2o], name='part3')

# ── part4 ── stage3 conv_b + stage3 skip + head ──  in: c3, m2 → out: DMA (logits, pre-softmax)
p4_c3 = tf.keras.Input((8, 8, 64), name='p4_c3')
p4_m2 = tf.keras.Input((16, 16, 32), name='p4_m2')
_x7  = L('conv2d_7')(p4_c3)
_x8  = L('conv2d_8')(p4_m2)                                 # stage3 1x1 skip (conv+BN)
_out = L('dense')(L('global_average_pooling2d')(L('q_activation_6')(L('add_2')([_x7, _x8]))))
part4 = tf.keras.Model([p4_c3, p4_m2], _out, name='part4')

for p in (part1, part2, part3, part4):
    print(f'{p.name}: in {[tuple(t.shape[1:]) for t in p.inputs]}'
          f' -> out {[tuple(t.shape[1:]) for t in p.outputs]}')

/home/tanawin/miniconda3/envs/dfx4ml-hls4ml/lib/python3.10/site-packages/keras/src/constraints.py:365: UserWarning: The `keras.constraints.serialize()` API should only be used for objects of type `keras.constraints.Constraint`. Found an instance of type <class 'qkeras.quantizers.quantized_bits'>, which may lead to improper serialization.
  warnings.warn(


full model params: 78723
part1: in [(32, 32, 3)] -> out [(32, 32, 16), (32, 32, 16)]
part2: in [(32, 32, 16), (32, 32, 16)] -> out [(16, 16, 32), (16, 16, 32)]
part3: in [(16, 16, 32), (16, 16, 32)] -> out [(8, 8, 64), (16, 16, 32)]
part4: in [(8, 8, 64), (16, 16, 32)] -> out [(10,)]


### 2b. Per-layer reuse factors + resource-balance check
`REUSE_FACTORS` starts from `exp_sz/hls_cls_cnn_q.py` and is **rebalanced** (conv2d_2/3/4
raised) so the four kernels are equal in area. The cell prints the per-partition
`eff = Σ weights/RF` and the resulting **region sizes** (region = max of its two RMs).

In [3]:
# per-layer ReuseFactor — rebalanced for the QUANTIZED model so the 4 RMs are as equal as
# possible in area. Weights are 8-bit uniform, so eff = W/RF is still the area proxy. Starting
# from the float RF map, conv2d_2/3/4 are raised (legal hls4ml divisors) to flatten part2/part3:
#   conv2d_2: 8 -> 9    (eff 288 -> 256)
#   conv2d_3: 24 -> 36  (eff 192 -> 128)   [RF must be a legal divisor; 32 is not, 36 is]
#   conv2d_4: 24 -> 36  (eff 384 -> 256)
# Result: parts [432, 416, 448, 426]; regions A=max(p1,p3)=448, B=max(p2,p4)=426 (was 576/512).
DEFAULT_REUSE = 1
REUSE_FACTORS = {
    'conv2d':    3,    # stem          432 w  -> eff 144
    'conv2d_1':  8,    # stage1 conv_a 2304 w -> eff 288
    'conv2d_2':  9,    # stage1 conv_b 2304 w -> eff 256  (was RF 8)
    'conv2d_3':  36,   # stage2 conv_a 4608 w -> eff 128  (was RF 24)
    'conv2d_4':  36,   # stage2 conv_b 9216 w -> eff 256  (was RF 24)
    'conv2d_5':  16,   # stage2 skip    512 w -> eff  32
    'conv2d_6':  96,   # stage3 conv_a 18432 w-> eff 192
    'conv2d_7':  96,   # stage3 conv_b 36864 w-> eff 384
    'conv2d_8':  64,   # stage3 skip   2048 w -> eff  32
    'dense':     64,   # head           640 w -> eff  10
}
PRECISION = 'ap_fixed<16,6>'   # fallback only; per-layer precision is auto from QKeras quantizers
STRATEGY  = 'Resource'

# weight counts (MAC/cycle at RF=1) — from the hls_cls_cnn_q.py layer map
WEIGHTS = {'conv2d':432,'conv2d_1':2304,'conv2d_2':2304,'conv2d_3':4608,'conv2d_4':9216,
           'conv2d_5':512,'conv2d_6':18432,'conv2d_7':36864,'conv2d_8':2048,'dense':640}
PART_LAYERS = {
    'part1': ['conv2d', 'conv2d_1'],
    'part2': ['conv2d_2', 'conv2d_3', 'conv2d_5'],
    'part3': ['conv2d_4', 'conv2d_6'],
    'part4': ['conv2d_7', 'conv2d_8', 'dense'],
}
# region = max RM that loads into it (pblock must fit the largest); phases alternate regions
REGION_OF = {'part1': 'A', 'part3': 'A', 'part2': 'B', 'part4': 'B'}
print(f'  {"part":<6} {"eff (sum W/RF)":>14}  {"reg":>3}   layers')
print(f'  {"-"*56}')
total = 0
part_eff = {}
for part, layers in PART_LAYERS.items():
    eff = sum(WEIGHTS[n] / REUSE_FACTORS[n] for n in layers)
    part_eff[part] = eff
    total += eff
    print(f'  {part:<6} {eff:>14.0f}  {REGION_OF[part]:>3}   {layers}')
print(f'  {"-"*56}')
regA = max(part_eff['part1'], part_eff['part3'])
regB = max(part_eff['part2'], part_eff['part4'])
print(f'  total eff = {total:.0f}   ideal per-part = {total/4:.0f}   (DSP ~= 0.64*eff, LUT ~= 31*eff)')
print(f'  region A (part1|part3) = {regA:.0f}   region B (part2|part4) = {regB:.0f}   '
      f'fabric = {regA+regB:.0f}')

  part   eff (sum W/RF)  reg   layers
  --------------------------------------------------------
  part1             432    A   ['conv2d', 'conv2d_1']
  part2             416    B   ['conv2d_2', 'conv2d_3', 'conv2d_5']
  part3             448    A   ['conv2d_4', 'conv2d_6']
  part4             426    B   ['conv2d_7', 'conv2d_8', 'dense']
  --------------------------------------------------------
  total eff = 1722   ideal per-part = 430   (DSP ~= 0.64*eff, LUT ~= 31*eff)
  region A (part1|part3) = 448   region B (part2|part4) = 426   fabric = 874


## 3. Input pool (reproducible)
A seeded random pool standing in for normalised CIFAR-10 images (the trained model expects
per-channel mean/std-normalised inputs, ≈ N(0,1)). The board test compares the **hardware**
output to the **HLS csim** output, so the input distribution only needs to be reproducible;
set `USE_CIFAR = True` to use real CIFAR-10 test images instead (needs a one-time download).

In [4]:
USE_CIFAR = False        # True -> real CIFAR-10 test images (downloads ~170 MB once)
MAX_QUERIES = 1000
RNG_SEED = 42

if USE_CIFAR:
    (xtr, _), (xte, _) = tf.keras.datasets.cifar10.load_data()
    xtr = xtr.astype('float32') / 255.0
    xte = xte.astype('float32') / 255.0
    mean = xtr.mean(axis=(0, 1, 2), keepdims=True)
    std  = xtr.std(axis=(0, 1, 2),  keepdims=True) + 1e-7
    X_pool = ((xte - mean) / std)[:MAX_QUERIES].astype('float32')
else:
    X_pool = np.random.default_rng(RNG_SEED).standard_normal(
        (MAX_QUERIES, 32, 32, 3)).astype('float32')
print('input pool:', X_pool.shape, '| source:', 'CIFAR-10 test' if USE_CIFAR else f'seeded N(0,1) (seed={RNG_SEED})')

input pool: (1000, 32, 32, 3) | source: seeded N(0,1) (seed=42)


## 4. Tool paths
**Edit the two paths below** to point at *your* Vitis / Vivado 2023.2 installs (the dirs
that hold `settings64.sh`). The defaults (`/tools/Xilinx/...`) are placeholders.

In [5]:
# >>> EDIT THESE <<< Vitis / Vivado 2023.2 install dirs (each holds settings64.sh);
# Hls4ml_build.setup_env() sources them onto PATH at construction.
VITIS_PATH  = '/tools/Xilinx/Vitis/2023.2'
VIVADO_PATH = '/tools/Xilinx/Vivado/2023.2'

## 5. Topology (`CONFIG`)
One `Partition` per (region, rm). The four kernels map onto **two PR regions** (2 RMs each),
phases 0→3: `part1`(r0,phase0) → `part2`(r1,phase1) → `part3`(r0,phase2) → `part4`(r1,phase3).

Six inter-partition streams cross the cuts (each `region` tag is the producer's region — a
bank-packing hint; `alloc_phase`/`free_phase` are the producer/consumer phases). Tensors are
quantized 8-bit; `conv*` denotes the fused conv+BN output:

| stream | tensor | shape | producer→consumer | phases |
|---|---|---|---|---|
| a1    | relu(conv2d_1) | (32,32,16) | part1→part2 | 0→1 |
| skip0 | relu(conv2d)   | (32,32,16) | part1→part2 | 0→1 |
| c2    | relu(conv2d_3) | (16,16,32) | part2→part3 | 1→2 |
| skip2 | conv2d_5       | (16,16,32) | part2→part3 | 1→2 |
| c3    | relu(conv2d_6) | (8,8,64)   | part3→part4 | 2→3 |
| m2    | stage2 out (relu) | (16,16,32) | part3→part4 | 2→3 |

In [6]:
PART_TAG = 'resnet8_4part'      # output root / build_prj / export suffix

# config_4 — 4 partitions across 2 PR regions (2 RMs each). Stream order in each
# partition's `outputs` matches that partition model's output-port order.
# Every inter-partition stream is non-DMA io (it crosses the cut through a streamer BRAM,
# not the DMA), so we set depth=1 on each: the kernel's input/output hls::stream is just a
# pass-through to the streamer, no internal FIFO buffering needed. The DMA ports (part1 in,
# part4 out) carry no Stream and keep hls4ml's stamped pragma depth.
CONFIG = [
    Partition('part1', 'p_part1', part1, region=0, rm=0, inputs=[DMA], outputs=[
        Stream('a1',    region=0, alloc_phase=0, free_phase=1, depth=1),
        Stream('skip0', region=0, alloc_phase=0, free_phase=1, depth=1)]),
    Partition('part2', 'p_part2', part2, region=1, rm=0, inputs=['a1', 'skip0'], outputs=[
        Stream('c2',    region=1, alloc_phase=1, free_phase=2, depth=1),
        Stream('skip2', region=1, alloc_phase=1, free_phase=2, depth=1)]),
    Partition('part3', 'p_part3', part3, region=0, rm=1, inputs=['c2', 'skip2'], outputs=[
        Stream('c3',    region=0, alloc_phase=2, free_phase=3, depth=1),
        Stream('m2',    region=0, alloc_phase=2, free_phase=3, depth=1)]),
    Partition('part4', 'p_part4', part4, region=1, rm=1, inputs=['c3', 'm2'], outputs=[DMA]),
]
print('config_4:', [p.name for p in CONFIG])

config_4: ['part1', 'part2', 'part3', 'part4']


## 6. Run-stage toggles + build config
Flip these to control which stages run. `AMT_QUERY` must match the board notebook (staged
automatically in the last cell).

In [7]:
# ── run-stage toggles ──────────────────────────────────────
RUN_CSIM    = True      # end-to-end csim across the partitions (saves x / y / pred)
RUN_DIAG    = False     # OPTIONAL per-layer HLS csim bisect (slow)
# C-synthesis stage — pick ONE (mutually exclusive). 'fifo' supersedes 'synth'.
#   'none'  — skip synthesis entirely
#   'synth' — C-synthesis + ip_catalog package (needs Vitis)
#   'fifo'  — C-synthesis + FIFO-depth optimization (needs Vitis cosim — heavy)
SYNTH_MODE  = 'fifo'
RUN_HWBUILD = True      # dfx4ml Vivado hardware + PYNQ software build

# This ResNet operates on 32x32x3 images: the per-query feature maps are large, so the
# streamer banks hold relatively few queries (see min_total_query in the glue step). Keep
# AMT_QUERY modest; the csim over the full pool is the slow part on the host side.
AMT_QUERY = 256         # csim sample count — must match the board notebook's AMT_QUERY

# per-part output root — keeps artifacts from colliding with other quick-start parts.
BUILD_ROOT = REPO / 'hls4ml_dfx_out' / PART_TAG


# Hls4ml_build only stamps a single global ReuseFactor; this ResNet needs PER-LAYER reuse
# factors, so subclass it to overlay REUSE_FACTORS onto the generated hls4ml config.
class Hls4ml_build_perlayer(Hls4ml_build):
    def _cfg(self, model):
        c = super()._cfg(model)                       # sets Model Strategy/ReuseFactor/Precision
        for name, lcfg in c['LayerName'].items():
            if name in REUSE_FACTORS:
                lcfg['ReuseFactor'] = REUSE_FACTORS[name]
        return c


# shared Hls4ml_build construction config (everything except `partitions`)
HB_KW = dict(
    out_root       = BUILD_ROOT,
    board          = 'kv260',
    part           = 'xck26-sfvc784-2LV-c',
    clock_period   = '10ns',
    precision      = PRECISION,
    reuse_factor   = DEFAULT_REUSE,   # per-layer values overlaid by _cfg above
    strategy       = STRATEGY,
    total_banks    = 64,
    rm_index_width = 3,
    vitis_path     = VITIS_PATH,
    vivado_path    = VIVADO_PATH,
)

## 7. Construct the orchestrator

In [8]:
hb = Hls4ml_build_perlayer(partitions=CONFIG, **HB_KW)
print('inferred: amt_phase =', hb.amt_phase, '| num_regions =', hb.num_regions)

v++       -> /tools/Xilinx/Vitis/2023.2/bin/v++
vitis-run -> /tools/Xilinx/Vitis/2023.2/bin/vitis-run
vivado    -> /tools/Xilinx/Vivado/2023.2/bin/vivado
inferred: amt_phase = 3 | num_regions = 2


## 8. Convert (get the partial model)
Convert every partition to an hls4ml `ModelGraph` + firmware. hls4ml may bump a reuse
factor to the nearest legal divisor (it prints a `WARNING`); that is expected.

In [9]:
hb.convert_all()

================================================== convert part1


/home/tanawin/miniconda3/envs/dfx4ml-hls4ml/lib/python3.10/site-packages/keras/src/constraints.py:365: UserWarning: The `keras.constraints.serialize()` API should only be used for objects of type `keras.constraints.Constraint`. Found an instance of type <class 'qkeras.quantizers.quantized_bits'>, which may lead to improper serialization.
  warnings.warn(


================================================== convert part2
================================================== convert part3
================================================== convert part4
converted: ['part1', 'part2', 'part3', 'part4']


{'part1': <hls4ml.model.graph.ModelGraph at 0x7459983a29b0>,
 'part2': <hls4ml.model.graph.ModelGraph at 0x74599c0862f0>,
 'part3': <hls4ml.model.graph.ModelGraph at 0x74599c0762c0>,
 'part4': <hls4ml.model.graph.ModelGraph at 0x74599c086710>}

## 9. csim + save reference data
Runs the partitions end-to-end and saves `x_input.npy` / `y_keras.npy` / `y_pred_hls.npy`
to `BUILD_ROOT/_csim_data` (a sibling of the partition dirs that convert wipes).

In [10]:
CSIM_DATA_DIR = BUILD_ROOT / '_csim_data'
CSIM_DATA_DIR.mkdir(parents=True, exist_ok=True)

if RUN_CSIM:
    X_csim     = X_pool[:AMT_QUERY].astype(np.float32)                     # x  — board input
    y_keras    = np.asarray(full.predict(X_csim), dtype=np.float32)        # y  — float ref
    final, bus = hb.csim_chain(x0=X_csim, peek=10)                         # HLS csim chained out
    y_pred_hls = np.asarray(final, dtype=np.float32)                       # pred — board 'expected'

    np.save(CSIM_DATA_DIR / 'x_input.npy',    X_csim)
    np.save(CSIM_DATA_DIR / 'y_keras.npy',    y_keras)
    np.save(CSIM_DATA_DIR / 'y_pred_hls.npy', y_pred_hls)

    # quick top-1 agreement (HLS fixed-point vs float Keras)
    agree = float(np.mean(y_pred_hls.argmax(1) == y_keras.argmax(1)))
    print('saved ->', CSIM_DATA_DIR)
    print('  x_input.npy   ', X_csim.shape)
    print('  y_keras.npy   ', y_keras.shape)
    print('  y_pred_hls.npy', y_pred_hls.shape)
    print(f'  HLS vs Keras top-1 agreement: {agree:.3f}')

8/8 [==============================] - 1s 24ms/step
================================================================ part1
  in  DMA      (256, 32, 32, 3) first10= [ 0.3047 -1.0400  0.7505  0.9406 -1.9510 -1.3022  0.1278 -0.3162 -0.0168
 -0.8530]
  out a1       (256, 16384)   first10= [0.9961 0.0000 0.1250 0.0000 0.7266 0.0000 0.0039 0.0000 0.9961 0.0000]
  out skip0    (256, 16384)   first10= [0.4922 0.5469 0.0859 0.0000 0.0000 0.0000 0.0000 0.9961 0.8984 0.9961]
================================================================ part2
  in  a1       (256, 16384)   first10= [0.9961 0.0000 0.1250 0.0000 0.7266 0.0000 0.0039 0.0000 0.9961 0.0000]
  in  skip0    (256, 16384)   first10= [0.4922 0.5469 0.0859 0.0000 0.0000 0.0000 0.0000 0.9961 0.8984 0.9961]
  out c2       (256, 8192)    first10= [0.0000 0.9961 0.9961 0.9961 0.0000 0.9961 0.9961 0.9961 0.9961 0.0000]
  out skip2    (256, 8192)    first10= [ 1.2744  1.0000  7.5879  0.9102  1.4756 -5.7227  0.9834  1.8008  1.3057
  0.0332]
=====

## 10. (optional) per-layer diagnostic
Set `RUN_DIAG=True` for a per-layer fixed-point signal-collapse bisect. Safe to skip.

In [11]:
if RUN_DIAG:
    PROBE = ['conv2d','conv2d_1','conv2d_2','conv2d_3','conv2d_4','conv2d_5',
             'conv2d_6','conv2d_7','conv2d_8','dense']
    hb.diag_bisect(full, PROBE, X_pool[:2])

## 11. Streamer glue
Compute the dfx params (`-> hb.dfx`) and stitch the user-BD dispatcher TCL. **Note the
printed `lowest total_query`** — that is the streamer buffer capacity (queries per
iteration). For this 32×32 model it is small, so the board notebook must set
`set_amt_query_per_iter(...)` to **≤ that value** (the DFX manager iterates over `AMT_QUERY`).

In [12]:
hb.compute_streamer_glue()
hb.print_streamer_report()
QUERY_CAP = hb.dfx['report'].get('min_total_query')
print('\n>>> streamer per-iteration capacity (min_total_query) =', QUERY_CAP,
      '\n    on the board set set_amt_query_per_iter(...) <= this value.')

[dfx-streamer] streams:
  ────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
  | name       | shape              | region | alloc_phase | free_phase | precision | amt_entry_per_query | bits_per_entry | amt_banks_per_entry | amt_query_per_bankGrp |
  ────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
  | a1         | (32, 32, 16)       |      0 |           0 |          1 |        16 |                1024 |            256 |                   4 |                     4 |
  | skip0      | (32, 32, 16)       |      0 |           0 |          1 |        16 |                1024 |            256 |                   4 |                     4 |
  | c2         | (16, 16, 32)       |      1 |           1 |          2 |        16 |                 256 |            512 |                   8 |               

## 12. Synthesis + FIFO optimization

In [ ]:
assert SYNTH_MODE in ('none', 'synth', 'fifo'), f"SYNTH_MODE must be 'none'|'synth'|'fifo', got {SYNTH_MODE!r}"
if SYNTH_MODE == 'synth':
    hb.synth_all()
elif SYNTH_MODE == 'fifo':
    hb.synth_all(fifo_opt=True)
else:
    print('SYNTH_MODE = none — skipping synthesis')

================================================== fifo-opt part1


/home/tanawin/miniconda3/envs/dfx4ml-hls4ml/lib/python3.10/site-packages/keras/src/constraints.py:365: UserWarning: The `keras.constraints.serialize()` API should only be used for objects of type `keras.constraints.Constraint`. Found an instance of type <class 'qkeras.quantizers.quantized_bits'>, which may lead to improper serialization.
  warnings.warn(


FIFO optimization completed

****** v++ v2023.2 (64-bit)
  **** SW Build 4026344 on 2023-10-11-15:42:10
    ** Copyright 1986-2022 Xilinx, Inc. All Rights Reserved.
    ** Copyright 2022-2023 Advanced Micro Devices, Inc. All Rights Reserved.

Running Dispatch Server on port: 34257
INFO: [v++ 60-1548] Creating build summary session with primary output /media/tanawin/tanawin1701e/project8/dfx4ml/dfx4ml_code/hls4ml_dfx_out/resnet8_4part/part1/vitis_workspace/p_part1/vitis_unified_project/vitis_unified_project.hlscompile_summary, at Thu Jun 25 11:59:09 2026
INFO: [v++ 82-31] Launching vitis_hls: vitis_hls -nolog -run csynth -work_dir /media/tanawin/tanawin1701e/project8/dfx4ml/dfx4ml_code/hls4ml_dfx_out/resnet8_4part/part1/vitis_workspace/p_part1/vitis_unified_project -config /media/tanawin/tanawin1701e/project8/dfx4ml/dfx4ml_code/hls4ml_dfx_out/resnet8_4part/part1/hls_kernel_config.cfg -cmdlineconfig /media/tanawin/tanawin1701e/project8/dfx4ml/dfx4ml_code/hls4ml_dfx_out/resnet8_4part/part

## 13. Hardware build

In [ ]:
hw = None
if RUN_HWBUILD:
    hw = HwBuildHelper(
        build_folder_path  = f'./build_prj_{PART_TAG}',
        dfx_root_path      = '.',
        export_folder_path = f'./export_{PART_TAG}',
        req_gen_ip         = 1,
        num_core           = 4,
        clk_frq            = 99999001,
        test_mode          = 0,          # user kernels
        hls4ml_build       = hb,         # config pulled from hb
    )
    hw.run_build()
    hw.package_export_files()
    print('hardware build complete -> ./export_' + PART_TAG + '/hw')

## 14. Software build

In [ ]:
if RUN_HWBUILD:
    SwBuildHelper(hw_builder=hw).package_export_file()
    print('software build complete -> ./export_' + PART_TAG)

## 15. Stage data + board notebook into the export
Copies the csim reference arrays into `export/data/` and drops the matching board-side test
notebook (`hls4ml_2_region_4_rm.ipynb`) beside it, **patched** for this model's input/output
shapes (32×32×3 → 10) and `AMT_QUERY`. Set the per-iteration query count there to
`QUERY_CAP` (printed in the glue step) before running on the board.

In [ ]:
EXPORT_DIR  = REPO / f'export_{PART_TAG}'
EXPORT_DATA = EXPORT_DIR / 'data'
EXPORT_DATA.mkdir(parents=True, exist_ok=True)

for fname in ('x_input.npy', 'y_keras.npy', 'y_pred_hls.npy'):
    src = CSIM_DATA_DIR / fname
    if src.exists():
        shutil.copy(src, EXPORT_DATA / fname)
        print('copied', fname, '->', EXPORT_DATA / fname)

# stage the board notebook, patched for the ResNet-8 shapes (the template targets the toy
# 8x8x1 -> 4 model). Each replacement is guarded so a template change can't crash staging.
BOARD_NB = REPO / 'example' / 'tutorial' / 'hls4ml_2_region_4_rm.ipynb'
text = BOARD_NB.read_text()
patches = [
    ('INPUT_SHAPE     = (AMT_QUERY, 8,8,1)', 'INPUT_SHAPE     = (AMT_QUERY, 32,32,3)'),
    ('OUTPUT_SHAPE    = (AMT_QUERY, 4)',     'OUTPUT_SHAPE    = (AMT_QUERY, 10)'),
    ('AMT_QUERY       = 900',                f'AMT_QUERY       = {AMT_QUERY}'),
]
for old, new in patches:
    if old in text:
        text = text.replace(old, new)
        print('patched board nb:', new.strip())
    else:
        print('[warn] board-nb pattern not found (template changed?):', old)
dst = EXPORT_DIR / BOARD_NB.name
dst.write_text(text)
print('staged board notebook ->', dst)
print(f'\nREMINDER: on the board set set_amt_query_per_iter(...) <= QUERY_CAP (={QUERY_CAP!r}).')

## 16. Copy to the KV260 PYNQ board
Copy the packaged `export_resnet8_4part/` (hardware bitstreams, PYNQ drivers, csim data, and
the patched board notebook) to the KV260 **yourself**, then run the board notebook
(`hls4ml_2_region_4_rm.ipynb`) there — after setting the per-iteration query count to
`QUERY_CAP`.

From a terminal on this machine (KV260 PYNQ default login is `ubuntu`/`ubuntu`):

```bash
scp -r export_resnet8_4part/. ubuntu@<board-ip>:/home/ubuntu/jupyter_notebooks/resnet8_4part
```